### Scratch Implemetation of the GMLCQ algorithm

In [1]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

## Experimentation data loading

In [2]:
def generate_train_data(load_fn):
    x, y= load_fn(return_X_y = True)
    scaler = StandardScaler()
    x = scaler.fit_transform(x)
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=0)
    x_train, x_test = (torch.tensor(x_train, dtype=torch.float32), 
                       torch.tensor(x_test, dtype=torch.float32))
    y_train, y_test = (torch.tensor(y_train, dtype=torch.float32), 
                       torch.tensor(y_test, dtype=torch.float32))

    return x_train, x_test, y_train, y_test

x_train, x_test, y_train, y_test = generate_train_data(load_iris)
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape, sep= ' | ')

torch.Size([135, 4]) | torch.Size([135]) | torch.Size([15, 4]) | torch.Size([15])


## Prototype initalization

In [7]:
def init_prototypes(x:torch.Tensor, y: torch.Tensor, n_classes:int):
    prototypes, proto_labels = [], []
    for c in range(n_classes):
        idx = (y == c).nonzero(as_tuple= True)[0]
        proto = x[idx[0].clone()]
        proto = proto / (proto.sum() + 1e-8)
        prototypes.append(proto)
        # prototypes.append(x[idx[0]].clone())
        proto_labels.append(c)
    return torch.stack(prototypes), torch.tensor(proto_labels)

if __name__ == '__main__':
    n_classes = len(torch.unique(y_train))
    prototypes, proto_labels = init_prototypes(x_train, y_train, n_classes)
    print(f'Prototypes: {prototypes}', f'Labels: {proto_labels}', f'No classes: {n_classes}', sep='\n')


Prototypes: tensor([[ 0.1810, -0.1089,  0.1508, -0.1218, -0.0712,  0.0439,  0.1704, -0.2020,
          0.2164,  0.0414,  0.2190,  0.1893,  0.2915],
        [ 0.0657,  0.1293,  0.1869,  0.1194, -0.0027,  0.0650,  0.0917, -0.1999,
          0.2509, -0.0442, -0.0146,  0.2129,  0.1395],
        [-0.3004, -0.1187,  0.0759, -0.2347,  0.2134,  0.4725,  0.4226, -0.1210,
          0.3066, -0.6103,  0.3524,  0.4106,  0.1310]])
Labels: tensor([0, 1, 2])
No classes: 3


## Criteria Defintion for determing the optimal relevance matrix

In [8]:
class GMLVQLoss(nn.Module):
    def forward(self, d_correct: torch.Tensor, d_wrong: torch.Tensor) -> torch.Tensor:
        mu = (d_correct - d_wrong) / (d_correct + d_wrong + 1e-8)
        # return torch.sigmoid(mu).mean()
        return torch.relu(mu)

## GMLVQ model definition

In [9]:
class GMLVQ(nn.Module):

    def __init__(self, n_features: int, n_classes: int, lr_w = 0.01, lr_r=0.01, epochs=20):
        super().__init__()
        self.epochs = epochs
        self.n_classes = n_classes

        self.W = nn.Parameter(torch.empty(n_classes, n_features))
        self.r = nn.Parameter(torch.ones(n_features))
        nn.init.normal_(self.W, mean=0.0, std=0.1)

        self.optimizer = torch.optim.Adam(self.parameters(), lr=max(lr_w, lr_r))
        self.criterion = GMLVQLoss()
        self.W_labels: torch.Tensor | None = None

    def _distances(self, x: torch.Tensor) -> torch.Tensor:
        r = torch.clamp(self.r, min=1e-6)
        diff = x.unsqueeze(0) - self.W
        return (r * diff * diff).sum(dim=1)

    def fit(self, X:torch.Tensor, y:torch.Tensor):
        W_init, self.W_labels = init_prototypes(X, y, self.n_classes)
        with torch.no_grad():
            self.W.copy_(W_init)

        for epoch in range(self.epochs):
            epoch_loss = 0.0
            for i in range(len(X)):
                x, label = X[i], y[i].item()
                dists = self._distances(x)

                correct_mask = self.W_labels == label
                wrong_mask = ~correct_mask

                # weights = torch.softmax(-dists, dim=0)
                # d_correct = (weights[correct_mask] * dists[correct_mask]).sum()
                # d_wrong = (weights[wrong_mask] * dists[wrong_mask]).sum()
                d_correct = dists[correct_mask].min()
                d_wrong = dists[wrong_mask].min()

                loss = self.criterion(d_correct, d_wrong)
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                with torch.no_grad():
                    self.r.clamp(min=1e-6)
                epoch_loss += loss.item()
            if (epoch + 1) % 10 == 0:
                print(f'Epoch {epoch + 1}/{self.epochs} loss={epoch_loss / len(X):.4f}')

    @torch.no_grad()
    def predict(self, X:torch.Tensor) -> torch.Tensor:
        preds = []
        for x in X:
            dists = self._distances(x)
            preds.append(self.W_labels[dists.argmin()])
        return torch.stack(preds)

    @torch.no_grad()
    def score(self, X:torch.Tensor, y: torch.Tensor) -> float:
        return (self.predict(X) == y).float().mean().item()

    def get_relevance(self) -> torch.Tensor:
        return self.r.detach()

if __name__ == '__main__':
    from sklearn.datasets import load_iris, load_breast_cancer, load_wine
    load_fns = [load_iris, load_breast_cancer, load_wine]
    titles = ['Iris', 'Breast cancer', 'Wine']
    for i, load_fn in enumerate(load_fns):
        x_train, x_test, y_train, y_test = generate_train_data(load_fn)
        n_features, n_classes = x_train.shape[1], len(y_train.unique())
        model = GMLVQ(n_features=n_features, n_classes= n_classes, lr_w=0.005, lr_r=0.005, epochs=100)
        
        print(f'Training results for {titles[i]}', '='*26, sep='\n')
        model.fit(x_train, y_train)

        print(
            f'Train: {model.score(x_train, y_train)}',
            f'Test: {model.score(x_test, y_test)}',
            sep = '\t|\t'
        )

Training results for Iris
Epoch 10/100 loss=0.0041
Epoch 20/100 loss=0.0017
Epoch 30/100 loss=0.0019
Epoch 40/100 loss=0.0019
Epoch 50/100 loss=0.0022
Epoch 60/100 loss=0.0017
Epoch 70/100 loss=0.0007
Epoch 80/100 loss=0.0018
Epoch 90/100 loss=0.0015
Epoch 100/100 loss=0.0021
Train: 0.9777777791023254	|	Test: 1.0
Training results for Breast cancer
Epoch 10/100 loss=0.0009
Epoch 20/100 loss=0.0008
Epoch 30/100 loss=0.0008
Epoch 40/100 loss=0.0010
Epoch 50/100 loss=0.0009
Epoch 60/100 loss=0.0008
Epoch 70/100 loss=0.0008
Epoch 80/100 loss=0.0008
Epoch 90/100 loss=0.0007
Epoch 100/100 loss=0.0007
Train: 0.97265625	|	Test: 1.0
Training results for Wine
Epoch 10/100 loss=0.0001
Epoch 20/100 loss=0.0000
Epoch 30/100 loss=0.0000
Epoch 40/100 loss=0.0000
Epoch 50/100 loss=0.0000
Epoch 60/100 loss=0.0000
Epoch 70/100 loss=0.0000
Epoch 80/100 loss=0.0000
Epoch 90/100 loss=0.0000
Epoch 100/100 loss=0.0000
Train: 1.0	|	Test: 0.9444444179534912


# Omega Feautre relevance